# Segment 7 — Real-Time Optimization, Visualization & Benchmarking

This notebook implements the **Segment 7 prototype** for the Adaptive Variable Resolution 2.5D LiDAR Mapping system.

Pipeline:

**LiDAR → Preprocessing → Adaptive Downsampling → 2.5D Map → Static/Dynamic Layers → Profiling → Visualization → Benchmarking**

> The notebook uses synthetic LiDAR data by default so the complete pipeline can be tested before connecting a real `.bin` dataset.


In [ ]:
# Install dependencies
# Run this cell once if the packages are not already installed.

!pip install numpy matplotlib open3d psutil pandas


In [ ]:
# Imports

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import pandas as pd
import psutil
import time
from collections import defaultdict

print("Libraries loaded successfully.")


## 1. Generate / Load LiDAR Data

For initial testing, synthetic LiDAR frames are generated.

Later, replace this with the real LiDAR dataset loader.


In [ ]:
def generate_lidar_frame(num_points=100000, max_range=100):
    """
    Generate a synthetic LiDAR point cloud.

    Returns:
        points: Nx3 array containing [x, y, z].
    """

    angles = np.random.uniform(0, 2 * np.pi, num_points)
    distances = np.random.uniform(2, max_range, num_points)

    x = distances * np.cos(angles)
    y = distances * np.sin(angles)

    # Simulated ground
    z = np.random.normal(0, 0.05, num_points)

    # Simulated obstacle
    obstacle_mask = (
        (x > 10) & (x < 15) &
        (y > 2) & (y < 6)
    )

    z[obstacle_mask] += np.random.uniform(
        0.5, 2.0, obstacle_mask.sum()
    )

    points = np.column_stack((x, y, z))

    return points


In [ ]:
def load_kitti_bin(filename):
    """
    Load a KITTI-style LiDAR .bin file.

    Expected format:
        x, y, z, intensity
    """

    data = np.fromfile(filename, dtype=np.float32)

    points = data.reshape(-1, 4)

    xyz = points[:, :3]
    intensity = points[:, 3]

    return xyz, intensity


# Example:
# points, intensity = load_kitti_bin("data/000000.bin")


## 2. Preprocessing

Remove invalid points and points outside the mapping range.


In [ ]:
def preprocess_points(points, max_range=100):
    """Remove invalid and out-of-range points."""

    # Remove NaN / Inf
    mask = np.isfinite(points).all(axis=1)
    points = points[mask]

    # Horizontal distance from vehicle
    distance = np.sqrt(
        points[:, 0] ** 2 +
        points[:, 1] ** 2
    )

    # Keep points within range
    mask = distance <= max_range

    return points[mask]


## 3. Adaptive Voxel Downsampling

Resolution changes according to distance:

| Distance | Resolution |
|---|---:|
| 0–10 m | 5 cm |
| 10–30 m | 10 cm |
| 30–60 m | 25 cm |
| 60–100 m | 50 cm |

This reduces computation while keeping higher detail close to the vehicle.


In [ ]:
def adaptive_voxel_downsample(points):
    """
    Adaptive resolution:

    0-10m   -> 5cm
    10-30m  -> 10cm
    30-60m  -> 25cm
    60-100m -> 50cm
    """

    if len(points) == 0:
        return np.empty((0, 3))

    distance = np.sqrt(
        points[:, 0] ** 2 +
        points[:, 1] ** 2
    )

    resolution = np.zeros(len(points))

    resolution[distance < 10] = 0.05
    resolution[(distance >= 10) & (distance < 30)] = 0.10
    resolution[(distance >= 30) & (distance < 60)] = 0.25
    resolution[distance >= 60] = 0.50

    output = []

    for voxel_size in [0.05, 0.10, 0.25, 0.50]:

        mask = resolution == voxel_size

        if not np.any(mask):
            continue

        pts = points[mask]

        voxel_indices = np.floor(
            pts / voxel_size
        ).astype(np.int32)

        _, unique_indices = np.unique(
            voxel_indices,
            axis=0,
            return_index=True
        )

        output.append(pts[unique_indices])

    if not output:
        return np.empty((0, 3))

    return np.vstack(output)


## 4. Adaptive 2.5D Map

Each `(x, y)` cell stores:

- maximum observed height
- occupancy
- semantic label placeholder

The semantic layer can later be populated by Segment 3/4 outputs.


In [ ]:
class Adaptive25DMap:

    def __init__(self, size=100, resolution=0.5):

        self.size = size
        self.resolution = resolution

        self.cells = int(size / resolution)

        self.height = np.full(
            (self.cells, self.cells),
            np.nan
        )

        self.occupancy = np.zeros(
            (self.cells, self.cells),
            dtype=np.uint8
        )

        self.semantic = np.zeros(
            (self.cells, self.cells),
            dtype=np.int32
        )

    def update(self, points):

        half = self.size / 2

        for x, y, z in points:

            ix = int((x + half) / self.resolution)
            iy = int((y + half) / self.resolution)

            if (
                ix < 0 or
                iy < 0 or
                ix >= self.cells or
                iy >= self.cells
            ):
                continue

            if np.isnan(self.height[ix, iy]):
                self.height[ix, iy] = z
            else:
                self.height[ix, iy] = max(
                    self.height[ix, iy],
                    z
                )

            self.occupancy[ix, iy] = 1


## 5. Static and Dynamic Layers

Static information changes slowly, while dynamic information such as vehicles and pedestrians changes every frame.


In [ ]:
class StaticDynamicMap:

    def __init__(self, size=100):

        self.static_map = Adaptive25DMap(
            size=size,
            resolution=0.5
        )

        self.dynamic_map = Adaptive25DMap(
            size=size,
            resolution=0.5
        )

    def update_static(self, points):
        self.static_map.update(points)

    def update_dynamic(self, points):
        self.dynamic_map.update(points)


## 6. Performance Profiler

Measures:

- preprocessing time
- adaptive downsampling time
- mapping time
- total latency
- FPS
- memory usage


In [ ]:
class Profiler:

    def __init__(self):
        self.results = []

    def start(self):
        return time.perf_counter()

    def stop(self, start):
        return (time.perf_counter() - start) * 1000

    def record(
        self,
        frame_id,
        preprocessing,
        downsampling,
        mapping
    ):

        total = (
            preprocessing +
            downsampling +
            mapping
        )

        fps = 1000 / total if total > 0 else 0

        memory = (
            psutil.Process().memory_info().rss
            / (1024 * 1024)
        )

        self.results.append({
            "frame": frame_id,
            "preprocessing_ms": preprocessing,
            "downsampling_ms": downsampling,
            "mapping_ms": mapping,
            "total_ms": total,
            "FPS": fps,
            "memory_MB": memory
        })

    def dataframe(self):
        return pd.DataFrame(self.results)


## 7. Complete Single-Frame Pipeline

In [ ]:
def process_frame(
    points,
    map_system,
    profiler,
    frame_id
):

    # -------------------------
    # PREPROCESSING
    # -------------------------

    start = profiler.start()

    clean_points = preprocess_points(points)

    preprocessing_time = profiler.stop(start)

    # -------------------------
    # ADAPTIVE DOWNSAMPLING
    # -------------------------

    start = profiler.start()

    adaptive_points = adaptive_voxel_downsample(
        clean_points
    )

    downsampling_time = profiler.stop(start)

    # -------------------------
    # MAP UPDATE
    # -------------------------

    start = profiler.start()

    map_system.update_static(
        adaptive_points
    )

    mapping_time = profiler.stop(start)

    # -------------------------
    # RECORD PERFORMANCE
    # -------------------------

    profiler.record(
        frame_id,
        preprocessing_time,
        downsampling_time,
        mapping_time
    )

    return adaptive_points


## 8. Run the Pipeline

The following cell processes 10 synthetic LiDAR frames.


In [ ]:
map_system = StaticDynamicMap(size=100)
profiler = Profiler()

for frame_id in range(10):

    raw_points = generate_lidar_frame(
        num_points=100000
    )

    processed_points = process_frame(
        raw_points,
        map_system,
        profiler,
        frame_id
    )

    print(
        f"Frame {frame_id:02d}: "
        f"{len(raw_points):,} → "
        f"{len(processed_points):,} points"
    )


## 9. Performance Results

In [ ]:
results = profiler.dataframe()

display(results)


In [ ]:
print("Average latency:",
      results["total_ms"].mean(), "ms")

print("Minimum latency:",
      results["total_ms"].min(), "ms")

print("Maximum latency:",
      results["total_ms"].max(), "ms")

print("P95 latency:",
      np.percentile(results["total_ms"], 95), "ms")

print("Average FPS:",
      results["FPS"].mean())


## 10. FPS Visualization

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    results["frame"],
    results["FPS"],
    marker="o"
)

plt.xlabel("Frame")
plt.ylabel("FPS")
plt.title("Real-Time LiDAR Performance")
plt.grid()

plt.show()


## 11. Point Cloud Visualization

Open3D is used for the initial 3D visualization.


In [ ]:
def visualize_pointcloud(points):

    cloud = o3d.geometry.PointCloud()

    cloud.points = o3d.utility.Vector3dVector(
        points
    )

    o3d.visualization.draw_geometries(
        [cloud],
        window_name="Adaptive LiDAR"
    )


# Example:
# visualize_pointcloud(processed_points)


## 12. Uniform vs Adaptive Cell Count

The proposed system should eventually be benchmarked against a uniform-resolution map.


In [ ]:
def calculate_uniform_cells(
    size=100,
    resolution=0.05
):

    cells_per_axis = int(size / resolution)

    return cells_per_axis ** 2


def calculate_adaptive_cells(points):

    if len(points) == 0:
        return 0

    distance = np.sqrt(
        points[:, 0] ** 2 +
        points[:, 1] ** 2
    )

    cells = 0

    ranges = [
        (0, 10, 0.05),
        (10, 30, 0.10),
        (30, 60, 0.25),
        (60, 100, 0.50)
    ]

    for rmin, rmax, resolution in ranges:

        mask = (
            (distance >= rmin) &
            (distance < rmax)
        )

        if np.any(mask):

            pts = points[mask]

            voxels = np.floor(
                pts / resolution
            ).astype(np.int32)

            cells += len(
                np.unique(
                    voxels,
                    axis=0
                )
            )

    return cells


In [ ]:
raw = generate_lidar_frame(
    num_points=100000
)

adaptive = adaptive_voxel_downsample(raw)

uniform_cells = calculate_uniform_cells()

adaptive_cells = calculate_adaptive_cells(raw)

reduction = (
    1 -
    adaptive_cells / uniform_cells
) * 100

print("Uniform cells:", f"{uniform_cells:,}")
print("Adaptive cells:", f"{adaptive_cells:,}")
print(f"Cell reduction: {reduction:.2f}%")


## 13. Final Benchmark Summary

This produces a compact benchmark table for the current prototype.


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Average latency (ms)",
        "P95 latency (ms)",
        "Average FPS",
        "Peak memory (MB)",
        "Raw points",
        "Adaptive points",
        "Uniform cells",
        "Adaptive cells",
        "Cell reduction (%)"
    ],

    "Value": [
        results["total_ms"].mean(),
        np.percentile(results["total_ms"], 95),
        results["FPS"].mean(),
        results["memory_MB"].max(),
        len(raw),
        len(adaptive),
        uniform_cells,
        adaptive_cells,
        reduction
    ]
})

display(summary)


# Segment 7 Completion Checklist

The prototype now contains:

- [x] LiDAR input
- [x] Preprocessing
- [x] Adaptive voxel downsampling
- [x] 2.5D map
- [x] Static/dynamic map structure
- [x] Frame-by-frame pipeline
- [x] Latency measurement
- [x] FPS measurement
- [x] Memory measurement
- [x] P95 latency
- [x] Open3D visualization function
- [x] Uniform vs adaptive benchmark

### Next integration step

Connect:

**Segment 3 → Semantic Segmentation**

and

**Segment 4 → Object Detection/Tracking**

before using the system as the complete Adaptive 2.5D LiDAR pipeline.
